# M11.12 · Güven bölgesi yöntemleri — doğal gradyan, TRPO, PPO, GAE

Makine öğrenmesi problemleri optimizasyondur (en küçük kare hata, çapraz entropi, en çok olabilirlik, en büyük indirimli ödül). Doğru arama yöntemleri (yön bul, adım uzunluğu seç) ile güven bölgesi yöntemleri (yaklaşık modele güvendiğin bölgeyi seç, içinde en iyi noktayı bul): ikinci dereceden Taylor modeli, hiperküre bölgesi, Hessian pozitif yarı tanımlı değilse Cauchy noktası, dogleg ve bölge yarıçapı uyarlaması. Politikalarda mesafe parametrelerde değil dağılımlarda ölçülür: KL bölgesi, Fisher bilgi matrisi, doğal gradyan, eşlenik gradyan ile Fisher-vektör çarpımı, TRPO'nun monoton iyileşme sınırı ve çizgi araması, PPO'nun kırpılmış vekil amacı ve genelleştirilmiş avantaj kestirimi (GAE).

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m11-12-trust-region.html](https://m-gonul.github.io/pytorch-lab/m11-12-trust-region.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m11_12_trust_region.py
CHECKS = {
 "b": "import math\nimport numpy as np\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def derivs():\n        p = [0.3, -0.7]\n        h = 1e-6\n        g = need(ns, \"rosen_grad\")(p)\n        num = [(ns[\"rosen\"]([p[0] + h, p[1]]) - ns[\"rosen\"]([p[0] - h, p[1]])) / (2 * h), (ns[\"rosen\"]([p[0], p[1] + h]) - ns[\"rosen\"]([p[0], p[1] - h])) / (2 * h)]\n        close(np.array(g), np.array(num), rtol=1e-6, what=\"gradyan\")\n        H = np.array(need(ns, \"rosen_hess\")(p))\n        close(H, H.T, what=\"simetrik Hessian\")\n    R_.test(\"türevler\", derivs)\n\n    def dl():\n        H = [[2.0, 0.0], [0.0, 1.0]]\n        close(np.array(need(ns, \"dogleg\")([2.0, 1.0], H, 10.0)), np.array([-1.0, -1.0]), what=\"bölge geniş: Newton adımı\")\n        s = ns[\"dogleg\"]([2.0, 1.0], H, 0.5)\n        close(np.linalg.norm(s), 0.5, what=\"dar bölge: sınırda\")\n        s = ns[\"dogleg\"]([1.0, 0.0], [[-1.0, 0], [0, 1.0]], 0.3)\n        close(np.array(s), np.array([-0.3, 0.0]), what=\"negatif eğrilik: Cauchy noktası sınırda\")\n    R_.test(\"dogleg\", dl)\n\n    def cmp():\n        tr = need(ns, \"trust_region\")([-1.2, 1.0], 100)\n        gd = need(ns, \"gd_backtracking\")([-1.2, 1.0], 100)\n        dist = lambda p: math.hypot(p[0] - 1, p[1] - 1)\n        k_tr = next((i for i, p in enumerate(tr) if dist(p) < 1e-6), None)\n        ensure(k_tr is not None and k_tr < 60, \"güven bölgesi 60 yinelemede (1,1)'e ulaşmalı\")\n        ensure(dist(gd[-1]) > 0.05, \"gradyan inişi 100 yinelemede hâlâ uzakta\")\n        gd5 = ns[\"gd_backtracking\"]([-1.2, 1.0], 5000)\n        k_gd = next((i for i, p in enumerate(gd5) if dist(p) < 1e-3), None)\n        R_.table(\"Rosenbrock, başlangıç (−1.2, 1)\", [\"yöntem\", \"10. yineleme\", \"50.\", \"100.\", \"‖p − (1,1)‖ < 1e−3 için\"],\n                 [[\"güven bölgesi (dogleg, tam Hessian)\", f\"{dist(tr[10]):.2e}\", f\"{dist(tr[50]):.2e}\", f\"{dist(tr[100]):.2e}\", f\"{next(i for i, p in enumerate(tr) if dist(p) < 1e-3)}\"],\n                  [\"gradyan inişi + Armijo\", f\"{dist(gd[10]):.2e}\", f\"{dist(gd[50]):.2e}\", f\"{dist(gd[100]):.2e}\", f\"{k_gd if k_gd is not None else '> 5000'}\"]],\n                 note=\"Muz biçimli vadide gradyan yönü kötü koşullu: doğru araması küçük adımlara mahkûm. İkinci dereceden model eğriliği bilir; güven bölgesi modele ancak doğrulandığı mesafede güvenir.\")\n    R_.test(\"güven bölgesi vs doğru araması\", cmp)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngü kuralları\", lambda: (max_loops(src, 0, func=\"fisher\"), max_loops(src, 0, func=\"fvp\"), max_loops(src, 1, func=\"cg\"), max_loops(src, 0, func=\"natural_step\")))\n    rng = np.random.default_rng(0)\n    B, K = 200, 5\n    th = rng.normal(0, 1, (B, K))\n    r = rng.random((B, K))\n\n    def fis():\n        pi = need(ns, \"softmax\")(th)\n        F = need(ns, \"fisher\")(pi)\n        a = np.array([rng.choice(K, 20000, p=p) for p in pi[:3]])\n        for b in range(3):\n            sc = np.eye(K)[a[b]] - pi[b]\n            close(F[b], sc.T @ sc / 20000, atol=0.01, what=\"F = E[∇logπ ∇logπᵀ] (örneklem)\")\n        v = rng.normal(0, 1, (B, K))\n        close(need(ns, \"fvp\")(pi, v), np.einsum(\"bij,bj->bi\", F, v), what=\"matrissiz Fv\")\n        d = 1e-3 * rng.normal(0, 1, (B, K))\n        close(need(ns, \"kl\")(pi, ns[\"softmax\"](th + d)), 0.5 * np.einsum(\"bi,bij,bj->b\", d, F, d), rtol=0.01, atol=1e-12, what=\"KL ≈ ½ dᵀFd\")\n        g = need(ns, \"policy_grad\")(th, r)\n        h = 1e-6\n        num = np.stack([((ns[\"softmax\"](th + h * np.eye(K)[k]) * r).sum(1) - (ns[\"softmax\"](th - h * np.eye(K)[k]) * r).sum(1)) / (2 * h) for k in range(K)], 1)\n        close(g, num, atol=1e-8, what=\"politika gradyanı\")\n    R_.test(\"Fisher, KL, gradyan\", fis)\n\n    def nat():\n        pi = ns[\"softmax\"](th)\n        A = np.eye(K)[None] * 3 + 0.5 * np.ones((B, K, K))\n        bvec = rng.normal(0, 1, (B, K))\n        x = need(ns, \"cg\")(lambda v: np.einsum(\"bij,bj->bi\", A, v), bvec, 50)\n        close(x, np.linalg.solve(A, bvec[..., None])[..., 0], atol=1e-8, what=\"CG = doğrudan çözüm (toplu)\")\n        delta = 0.005\n        tn = need(ns, \"natural_step\")(th, r, delta)\n        g = ns[\"policy_grad\"](th, r)\n        tv = th + g * np.sqrt(2 * delta / np.einsum(\"bi,bi->b\", g, ns[\"fvp\"](pi, g)))[:, None]\n        kn, kv = ns[\"kl\"](pi, ns[\"softmax\"](tn)), ns[\"kl\"](pi, ns[\"softmax\"](tv))\n        ensure(np.abs(kn / delta - 1).max() < 0.15 and np.abs(kv / delta - 1).max() < 0.15, \"iki adım da KL ≈ δ\")\n        gain_n = (ns[\"softmax\"](tn) * r).sum(1) - (pi * r).sum(1)\n        gain_v = (ns[\"softmax\"](tv) * r).sum(1) - (pi * r).sum(1)\n        ensure((gain_n >= gain_v - 1e-9).mean() > 0.95, \"aynı KL'de doğal gradyan daha çok iyileştirir\")\n        s = ns[\"cg\"](lambda v: ns[\"fvp\"](pi, v) + 1e-3 * v, g)\n        adv = r - (pi * r).sum(1, keepdims=True)\n        cos = (s * (adv - adv.mean(1, keepdims=True))).sum(1) / (np.linalg.norm(s, axis=1) * np.linalg.norm(adv - adv.mean(1, keepdims=True), axis=1))\n        R_.table(f\"{B} softmax haydut, KL bütçesi δ = {delta}\", [\"ölçü\", \"doğal gradyan\", \"düz gradyan (aynı KL)\"],\n                 [[\"ort. KL / δ\", f\"{kn.mean() / delta:.3f}\", f\"{kv.mean() / delta:.3f}\"], [\"ort. beklenen ödül artışı\", f\"{gain_n.mean():.5f}\", f\"{gain_v.mean():.5f}\"],\n                  [\"F⁻¹g ile (avantaj − ort.) arasındaki kosinüs\", f\"{cos.mean():.4f}\", \"\"]],\n                 note=\"Softmax'ta Fisher tekildir (θ'ya sabit eklemek politikayı değiştirmez); sönümleme bunu çözer. Doğal gradyan, merkezlenmiş avantajın kendisidir: parametrelemeden bağımsız 'en dik' yön.\")\n    R_.test(\"doğal gradyan\", nat)\n",
 "d": "import torch\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def gl():\n        r = torch.tensor([[1.0], [1.0], [1.0]])\n        v = torch.tensor([[0.5], [0.2], [0.1]])\n        d = torch.tensor([[0.0], [1.0], [0.0]])\n        adv, ret = need(ns, \"gae\")(r, v, d, torch.tensor([2.0]), 0.9, 0.8)\n        d0 = 1 + 0.9 * 0.2 - 0.5\n        d1 = 1 - 0.2\n        d2 = 1 + 0.9 * 2.0 - 0.1\n        close(adv[:, 0].numpy(), torch.tensor([d0 + 0.72 * d1, d1, d2]).numpy(), what=\"GAE (bitişte kesilir)\")\n        close(ret.numpy(), (adv + v).numpy(), what=\"dönüş hedefi = A + V\")\n        lp = torch.log(torch.tensor([0.6, 0.2]))\n        lo = torch.log(torch.tensor([0.4, 0.4]))\n        close(float(need(ns, \"ppo_loss\")(lp, lo, torch.tensor([2.0, -1.0]), 0.2)), -(1.2 * 2 + (-0.8)) / 2, what=\"kırpılmış amaç\")\n    R_.test(\"gae ve ppo_loss\", gl)\n\n    def fis():\n        torch.manual_seed(0)\n        pi = torch.nn.Sequential(torch.nn.Linear(3, 4), torch.nn.Tanh(), torch.nn.Linear(4, 2))\n        X = torch.randn(50, 3)\n        params = list(pi.parameters())\n        n = sum(p.numel() for p in params)\n        probs = torch.softmax(pi(X), 1).detach()\n        J = []\n        for i in range(50):\n            for a in range(2):\n                gi = torch.autograd.grad(torch.log_softmax(pi(X[i:i + 1]), 1)[0, a], params)\n                J.append((probs[i, a], torch.cat([x.reshape(-1) for x in gi])))\n        F = sum(p * torch.outer(g, g) for p, g in J) / 50\n        v = torch.randn(n)\n        close(need(ns, \"fvp\")(pi, X, v, 0.0).numpy(), (F @ v).numpy(), atol=1e-5, what=\"Fisher-vektör çarpımı = E[∇logπ∇logπᵀ]v\")\n        A = F + 0.1 * torch.eye(n)\n        b = torch.randn(n)\n        x = need(ns, \"cg\")(lambda u: A @ u, b, 200)\n        close(x.numpy(), torch.linalg.solve(A, b).numpy(), atol=1e-4, what=\"CG\")\n    R_.test(\"fvp ve cg\", fis)\n\n    def step():\n        torch.manual_seed(1)\n        g = torch.Generator().manual_seed(1)\n        net = ns[\"PolicyValue\"]()\n        state = [(torch.rand(16, 4, generator=g) - 0.5) * 0.1, torch.zeros(16), []]\n        X, A, LP, V, R, D, vl = ns[\"rollout\"](net, state, g)\n        adv, _ = ns[\"gae\"](R, V, D, vl, 0.99, 0.95)\n        X, A, LP, adv = X.reshape(-1, 4), A.reshape(-1), LP.reshape(-1), adv.reshape(-1)\n        adv = (adv - adv.mean()) / adv.std()\n        p_old = torch.softmax(net.pi(X), 1).detach()\n        L0 = float((adv * 1.0).mean())\n        ok = need(ns, \"trpo_step\")(net.pi, X, A, adv, LP, 0.01)\n        with torch.no_grad():\n            kl = float((p_old * (torch.log(p_old) - torch.log_softmax(net.pi(X), 1))).sum(1).mean())\n            L1 = float((torch.exp(torch.log_softmax(net.pi(X), 1).gather(1, A[:, None]).squeeze(1) - LP) * adv).mean())\n        ensure(ok and kl <= 0.015 and L1 > L0, f\"TRPO adımı: KL {kl:.4f} ≤ 1.5δ ve vekil artmalı ({L0:.4f} → {L1:.4f})\")\n    R_.test(\"trpo_step\", step)\n\n    def train():\n        npp, lp = need(ns, \"train_ppo\")(60, seed=0)\n        nt, lt = need(ns, \"train_trpo\")(60, seed=0)\n        ep, et = float(ns[\"evaluate\"](npp).mean()), float(ns[\"evaluate\"](nt).mean())\n        ensure(ep >= 200 and et >= 200, f\"PPO ve TRPO CartPole'u öğrenmeli ({ep:.0f}, {et:.0f})\")\n        R_.table(\"CartPole, 60 yineleme × 16 ortam × 128 adım (122 880 adım)\", [\"yöntem\", \"son 10 eğitim bölümü\", \"açgözlü değerlendirme\"],\n                 [[\"PPO (ε = 0.2, 4 epoch)\", f\"{sum(lp[-10:]) / 10:.0f}\", f\"{ep:.0f}\"], [\"TRPO (δ = 0.01)\", f\"{sum(lt[-10:]) / 10:.0f}\", f\"{et:.0f}\"]],\n                 note=\"Aynı veri bütçesinde A2C'nin üçte biri kadar adım. PPO her veriyi birkaç epoch kullanır (kırpma güvenli kılar); TRPO her yinelemede tek ama 'optimal büyüklükte' bir adım atar.\")\n    R_.test(\"train_ppo ve train_trpo\", train)\n",
 "e": "import torch\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef env_loop(r, v, d, v_last, gamma, lam):\n    T, N = r.shape\n    out = [[0.0] * N for _ in range(T)]\n    rl, vl, dl, ll = r.tolist(), v.tolist(), d.tolist(), v_last.tolist()\n    for n in range(N):\n        last = 0.0\n        for t in range(T - 1, -1, -1):\n            nv = ll[n] if t == T - 1 else vl[t + 1][n]\n            last = rl[t][n] + gamma * (1 - dl[t][n]) * nv - vl[t][n] + gamma * lam * (1 - dl[t][n]) * last\n            out[t][n] = last\n    return out\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"gae_matrix döngüsüz\", lambda: (max_loops(src, 0, func=\"gae_matrix\"), max_loops(src, 1, func=\"gae_scan\")))\n    g = torch.Generator().manual_seed(0)\n\n    def eq():\n        rows = []\n        for T, N in ((16, 64), (128, 64), (512, 16)):\n            r = torch.randn(T, N, generator=g, dtype=torch.float64)\n            v = torch.randn(T, N, generator=g, dtype=torch.float64)\n            d = (torch.rand(T, N, generator=g) < 0.05).double()\n            vl = torch.randn(N, generator=g, dtype=torch.float64)\n            a1 = need(ns, \"gae_scan\")(r, v, d, vl, 0.99, 0.95)\n            a2 = need(ns, \"gae_matrix\")(r, v, d, vl, 0.99, 0.95)\n            close(a1.numpy(), torch.tensor(env_loop(r, v, d, vl, 0.99, 0.95), dtype=torch.float64).numpy(), atol=1e-10, what=\"tarama = referans\")\n            close(a2.numpy(), a1.numpy(), atol=1e-10, what=\"matris = tarama\")\n            t0 = bench(lambda: env_loop(r, v, d, vl, 0.99, 0.95), repeat=1, number=1)\n            t1 = bench(lambda: ns[\"gae_scan\"](r, v, d, vl, 0.99, 0.95), repeat=3, number=1)\n            t2 = bench(lambda: ns[\"gae_matrix\"](r, v, d, vl, 0.99, 0.95), repeat=3, number=1)\n            rows.append([f\"{T} × {N}\", fmt_time(t0), fmt_time(t1), fmt_time(t2)])\n        R_.table(\"GAE süreleri\", [\"T × N\", \"saf Python (ortam × zaman)\", \"zaman taraması, ortamlar paralel\", \"(N, T, T) matris\"], rows,\n                 note=\"Matris sürümü O(T²N) iş yapar: kısa T'de Python döngüsünü yok ettiği için hızlı, uzun T'de gereksiz sıfırlarla yavaş. Doğru vektörleştirme bağımlılık zincirini (zaman) sıralı bırakıp bağımsız ekseni (ortamlar) paralelleştirir.\")\n    R_.test(\"gae_scan = gae_matrix\", eq)\n\n    def lam():\n        S, R, D = need(ns, \"walk_batch\")(20000, 400, torch.Generator().manual_seed(1))\n        ensure(S.shape == (401, 20000) and float(D[-1].mean()) > 0.98, \"şekil ve bitiş\")\n        true = torch.linspace(-1, 1, 21, dtype=torch.float64)\n        true[0] = true[-1] = 0\n        Vh = true + 0.3 * torch.sin(torch.arange(21, dtype=torch.float64) * 1.3)\n        Vh[0] = Vh[-1] = 0\n        V = Vh[S]\n        rows, mse = [], {}\n        for l in (0.0, 0.5, 0.8, 0.9, 0.95, 1.0):\n            a = ns[\"gae_scan\"](R, V[:-1], D, V[-1], 1.0, l)\n            ret0 = a[0] + V[0]\n            mse[l] = float((ret0 ** 2).mean())\n            rows.append([l, f\"{float(ret0.mean()):+.4f}\", f\"{float(ret0.var()):.4f}\", f\"{mse[l]:.4f}\"])\n        best = min(mse, key=mse.get)\n        ensure(0 < best < 1 and mse[best] < 0.5 * mse[1.0], f\"ara λ en iyi (bulunan {best})\")\n        R_.table(\"19 durumlu yürüyüş, merkezde λ-getirisi hedefi (gerçek değer 0), V̂ = V + 0.3 sin(1.3 s), 20 000 bölüm\", [\"λ\", \"ortalama\", \"varyans\", \"MSE\"], rows,\n                 note=\"λ küçükken hedef yanlış V̂'ye dayanır (yanlılık ve V̂ farklarından gelen gürültü); λ = 1'de tüm bölüm rastgeleliği hedefe girer. GAE makalesindeki λ ≈ 0.95–0.99 seçimi bu ödünleşimin uzun ufuklu görevlerdeki yeridir.\")\n    R_.test(\"λ ödünleşimi\", lam)\n",
 "x1": "import math\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        nw = need(ns, \"newton\")([1.0, 0.01], 20)\n        tr = need(ns, \"trust_region_saddle\")([1.0, 0.01], 60)\n        fN, fT = ns[\"f\"](nw[-1]), ns[\"f\"](tr[-1])\n        ensure(math.hypot(*nw[-1]) < 1e-6, \"Newton eyer noktasına (0,0) gitmeli\")\n        ensure(abs(fT + 1) < 1e-6 and abs(abs(tr[-1][1]) - math.sqrt(2)) < 1e-4, \"güven bölgesi minimuma (0, ±√2)\")\n        R_.table(\"f = x² − y² + y⁴/4, başlangıç (1, 0.01)\", [\"yöntem\", \"son nokta\", \"f\"], [[\"Newton\", f\"({nw[-1][0]:.2e}, {nw[-1][1]:.2e})\", f\"{fN:.4f}\"], [\"güven bölgesi\", f\"({tr[-1][0]:.2e}, {tr[-1][1]:.4f})\", f\"{fT:.4f}\"]],\n                 note=\"Newton ikinci dereceden modelin durağan noktasına gider — H tanımsızsa bu eyer olabilir. Güven bölgesi modeli en aza indirir: negatif eğrilik yönünde bölge sınırına kadar gider ve eyerden kaçar.\")\n    R_.test(\"eyer noktası\", t)\n",
 "x2": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        tg = np.linspace(-2, 4, 601)\n        th0 = np.array([0.2, -0.1, 0.0])\n        d = np.array([1.0, -0.6, -0.2])\n        A = np.array([1.0, -0.8, 0.1])\n        cpi, clip, klp, kl = need(ns, \"surrogates\")(tg, th0, d, A)\n        ensure(np.all(clip <= cpi + 1e-12), \"L^CLIP ≤ L^CPI (kötümser)\")\n        i0 = np.argmin(np.abs(tg))\n        close(clip[i0], cpi[i0], what=\"t = 0'da eşit\")\n        h = tg[1] - tg[0]\n        close((clip[i0 + 1] - clip[i0 - 1]) / (2 * h), (cpi[i0 + 1] - cpi[i0 - 1]) / (2 * h), rtol=1e-3, what=\"t = 0'da aynı gradyan\")\n        tc, tk, tp = tg[clip.argmax()], tg[klp.argmax()], tg[cpi.argmax()]\n        ensure(tc < tp, \"L^CLIP'in en iyisi L^CPI'ninkinden önce\")\n        R_.table(\"Yön d boyunca vekil amaçlar (ε = 0.2, β = 1)\", [\"amaç\", \"en iyi t\", \"orada KL\", \"orada L^CPI\"],\n                 [[\"L^CPI (kısıtsız)\", f\"{tp:.2f}\", f\"{kl[cpi.argmax()]:.3f}\", f\"{cpi.max():.4f}\"], [\"L^CLIP\", f\"{tc:.2f}\", f\"{kl[clip.argmax()]:.3f}\", f\"{cpi[clip.argmax()]:.4f}\"],\n                  [\"L^CPI − β·KL\", f\"{tk:.2f}\", f\"{kl[klp.argmax()]:.3f}\", f\"{cpi[klp.argmax()]:.4f}\"]],\n                 note=\"L^CPI tek başına bu yönde sınırsız ilerlemeyi ödüllendirir (aralığın ucunda en büyük); kırpma ve KL cezası adımı politikaya yakın bir bölgede tutar (PPO makalesi Şekil 2).\")\n    R_.test(\"vekil amaçlar\", t)\n",
 "x3": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rng = np.random.default_rng(3)\n        S, A, gamma = 6, 3, 0.9\n        P = rng.random((A, S, S)) ** 2\n        P /= P.sum(-1, keepdims=True)\n        R = rng.random((S, A))\n        d0 = np.full(S, 1 / S)\n        pi = rng.dirichlet(np.ones(A), S)\n        pi2 = rng.dirichlet(np.ones(A), S)\n        e0 = need(ns, \"eta\")(P, R, d0, pi, gamma)\n        close(need(ns, \"surrogate_L\")(P, R, d0, pi, pi, gamma), e0, what=\"L_π(π) = η(π)\")\n        rows, viol = [], 0\n        for a in (0.0, 0.01, 0.05, 0.1, 0.3, 0.6, 1.0):\n            pn = (1 - a) * pi + a * pi2\n            e, Lv, b = ns[\"eta\"](P, R, d0, pn, gamma), ns[\"surrogate_L\"](P, R, d0, pi, pn, gamma), need(ns, \"bound\")(P, R, d0, pi, pn, gamma)\n            viol += int(b > e + 1e-12)\n            rows.append([a, f\"{e:.5f}\", f\"{Lv:.5f}\", f\"{b:.4f}\", f\"{need(ns, 'kl_max')(pi, pn):.5f}\"])\n        ensure(viol == 0, \"η ≥ sınır her karışımda\")\n        h = 1e-6\n        de = (ns[\"eta\"](P, R, d0, (1 - h) * pi + h * pi2, gamma) - e0) / h\n        dL = (ns[\"surrogate_L\"](P, R, d0, pi, (1 - h) * pi + h * pi2, gamma) - e0) / h\n        close(de, dL, rtol=1e-4, what=\"L ve η birinci mertebede aynı\")\n        R_.table(\"π̃ = (1 − α)π + απ′ (S = 6, A = 3, γ = 0.9)\", [\"α\", \"η(π̃)\", \"L_π(π̃)\", \"alt sınır\", \"KL^max\"], rows,\n                 note=\"Sınır doğru ama çok kötümser: (1 − γ)⁻² = 100 çarpanı yüzünden yalnız minik adımlarda anlamlı. TRPO bu yüzden ceza yerine KL kısıtı kullanır ve δ'yı deneysel seçer.\")\n    R_.test(\"monoton iyileşme sınırı\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m11_12_trust_region as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Optimizasyon olarak öğrenme:** en küçük kare hata, en küçük çapraz entropi, en büyük olabilirlik, en büyük indirimli ödül. İki aile:

- **Doğru arama:** iyileşme yönü bul (gradyan), adım uzunluğu seç (geri izleme, Armijo).
- **Güven bölgesi:** $f$'yi basit bir $\tilde f$ ile yaklaşıkla, $\tilde f$'ye güvendiğin bölge $R_t$ seç, $\tilde x^*=\arg\min_{x\in R_t}\tilde f(x)$.

$\tilde f$ ikinci dereceden Taylor: $\tilde f(x)=f(c)+\nabla f^\top(x-c)+\tfrac12(x-c)^\top H(x-c)$ — $H$ $d_x\times d_x$: büyük modellerde sorun. Taylor $c$ yakınında doğru → bölge hiperküre $\|x-c\|\le\delta$. $H$ pozitif yarı tanımlıysa dışbükey alt problem (basit, global çözüm); değilse iyileşmeyi garanti eden sezgiseller (Cauchy noktası: gradyan yönünde bölge içinde en iyi nokta; dogleg; Steihaug-CG). Oran $\rho=\dfrac{f(x)-f(x+p)}{\tilde f(x)-\tilde f(x+p)}$ yarıçapı yönetir.

**Politikalarda:** mesafeyi davranışla ölç: $\mathrm{KL}(\pi_\theta\|\pi_{\theta+d})\approx\tfrac12d^\top F d$, $F=\mathbb E[\nabla\log\pi\,\nabla\log\pi^\top]$. KL bölgesinde doğrusal amacın çözümü **doğal gradyan** $F^{-1}g$ (Kakade 2001), adım $\beta=\sqrt{2\delta/(g^\top F^{-1}g)}$.

**TRPO:** vekil amaç $L_\pi(\tilde\pi)=\mathbb E_{s,a\sim\pi}\big[\tfrac{\tilde\pi(a|s)}{\pi(a|s)}A_\pi(s,a)\big]$; teorem $\eta(\tilde\pi)\ge L_\pi(\tilde\pi)-C\,\mathrm{KL}^{\max}$, $C=4\epsilon\gamma/(1-\gamma)^2$ → sınırı her adımda iyileştirmek monoton iyileşme verir. Pratikte: $\max L$ s.t. ortalama KL ≤ δ; $s=F^{-1}g$ eşlenik gradyanla (Fisher-vektör çarpımı = KL'nin Hessian-vektör çarpımı, iki kez geri yayılım), sonra KL ve iyileşme koşullu geri izlemeli çizgi araması.

**PPO:** ikinci derece hesap yok; oran $r_t=\pi_\theta/\pi_{\text{eski}}$, amaç $\mathbb E[\min(r_tA_t,\mathrm{clip}(r_t,1-\epsilon,1+\epsilon)A_t)]$ — kötümser alt sınır; aynı veride birkaç epoch mini-batch SGD. **GAE:** $\delta_t=r_t+\gamma V(s_{t+1})-V(s_t)$, $A_t=\sum_k(\gamma\lambda)^k\delta_{t+k}$: λ = 0 TD (yanlı, düşük varyans), λ = 1 MC (yansız, yüksek varyans).

## 1 · Elle

1-B: kısıtsız minimum $-g/H$: H = 1 → −2, bölge dışında → sınır −1. H = 4 → −0.5, içeride.

Doğal adım: $F^{-1}g=(0.25,0)$, $g^\top F^{-1}g=0.25$, $\beta=\sqrt{0.04/0.25}=0.4$ → $d=(0.1,0)$; KL ≈ ½·4·0.01 = 0.02 = δ ✓.

Kırpma: $\min(3,\,1.2\cdot2=2.4)=2.4$; $\min(-0.5,\,0.8\cdot(-1)=-0.8)=-0.8$ — her iki durumda daha kötümser değer.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**1-B güven bölgesi: $\tilde f(x)=f(c)+g(x-c)+\tfrac12H(x-c)^2$, $|x-c|\le1$, $g=2$**

- `tr1`: H = 1: adım $x-c$
- `tr2`: H = 4: adım $x-c$

**Politika mesafeleri**

- `kl`: KL((0.5, 0.5) ‖ (0.6, 0.4))
- `nb`: Doğal gradyan adımı: g = (1,0), F = diag(4,1), δ = 0.02 → $\beta=\sqrt{2\delta/g^\top F^{-1}g}$

**PPO ve GAE (ε = 0.2)**

- `c1`: r = 1.5, A = 2: kırpılmış amaç
- `c2`: r = 0.5, A = −1: kırpılmış amaç
- `gae`: δ₀ = 1, δ₁ = 0.5 (sonra bitiş), γ = 0.9, λ = 0.8: $A_0$

In [ ]:
cevaplar = dict(
    tr1=None,
    tr2=None,
    kl=None,
    nb=None,
    c1=None,
    c2=None,
    gae=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"tr1":[-1,0],"tr2":[-0.5,0],"kl":[0.020410997260127586,1e-9],"nb":[0.4,0],"c1":[2.4,0],"c2":[-0.8,0],"gae":[1.36,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, 2 boyut. **Rosenbrock** $f=(1-x)^2+100(y-x^2)^2$, başlangıç (−1.2, 1), minimum (1, 1).

- `rosen(p)`, `rosen_grad(p)`, `rosen_hess(p)` (2×2 liste).
- `gd_backtracking(p0, n_iter)` — gradyan inişi, Armijo geri izleme ($t\leftarrow t/2$, $f(p-tg)\le f(p)-10^{-4}t\|g\|^2$, $t_0=1$); nokta listesi.
- `dogleg(g, H, delta)` — H pozitif tanımlıysa dogleg; değilse Cauchy noktası $p=-\tau\,\delta\,g/\|g\|$, $\tau=1$ ($g^\top Hg\le0$) ya da $\min(\|g\|^3/(\delta g^\top Hg),1)$.
- `trust_region(p0, n_iter, delta0=1.0, delta_max=10.0, eta=0.1)` — ρ < 0.25 → δ/4; ρ > 0.75 ve adım sınırda → min(2δ, δ_max); ρ > η → kabul. Nokta listesi.

In [ ]:
%%writefile stage_b.py
import math


def rosen(p):
    ...


def rosen_grad(p):
    ...


def rosen_hess(p):
    ...


def gd_backtracking(p0, n_iter):
    ...


def dogleg(g, H, delta):
    ...


def trust_region(p0, n_iter, delta0=1.0, delta_max=10.0, eta=0.1):
    ...


print(trust_region([-1.2, 1.0], 50)[-1], gd_backtracking([-1.2, 1.0], 50)[-1])

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

**Doğal gradyan**, NumPy, tek durumlu softmax politikalar (haydut), B problem aynı anda: `theta (B, K)`, ödüller `r (B, K)`.

- `softmax(theta)`; `kl(p, q)` → `(B,)`.
- `fisher(pi)` → `(B, K, K)`: $\mathrm{diag}(\pi)-\pi\pi^\top$.
- `fvp(pi, v)` → $Fv=\pi\odot v-\pi(\pi^\top v)$ (matrissiz).
- `policy_grad(theta, r)` → $\nabla_\theta\,\pi^\top r=\pi\odot(r-\pi^\top r)$.
- `cg(Avp, b, iters=20, tol=1e-12)` — eşlenik gradyan, `Avp` bir fonksiyon (döngü serbest; toplu: her satır ayrı sistem, vektörel adımlar).
- `natural_step(theta, r, delta, damping=1e-3)` — $s=\mathrm{CG}(F+\text{damping}\,I,\,g)$, $\beta=\sqrt{2\delta/s^\top Fs}$, $\theta+\beta s$.

In [ ]:
%%writefile stage_c.py
import numpy as np


def softmax(theta):
    ...


def kl(p, q):
    ...


def fisher(pi):
    ...


def fvp(pi, v):
    ...


def policy_grad(theta, r):
    ...


def cg(Avp, b, iters=20, tol=1e-12):
    ...


def natural_step(theta, r, delta, damping=1e-3):
    ...


th = np.zeros((1, 3)); r = np.array([[1.0, 0.0, 0.5]])
print(natural_step(th, r, 0.01))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

**TRPO ve PPO**, CartPole, 16 paralel ortam × 128 adım. Ortak: `PolicyValue` (ayrı politika ve değer MLP'leri, tanh), `rollout(...)` (verildi), `gae(r, v, d, v_last, gamma, lam)`.

- `ppo_loss(logp_new, logp_old, adv, eps)` → $-\overline{\min(r A,\mathrm{clip}(r)A)}$.
- `train_ppo(iters=60, eps=0.2, epochs=4, mb=256, lr=3e-4, seed=0)` — avantaj normalize; kayıp = PPO + 0.5·değer MSE; gradyan normu 0.5.
- `fvp(pi, X, v, damping=0.1)` — ortalama $\mathrm{KL}(\pi_{\text{sabit}}\|\pi_\theta)$'nın Hessian'ı × v (iki kez `autograd.grad`).
- `cg(Avp, b, iters=10)`.
- `trpo_step(pi, X, A, adv, old_logp, max_kl=0.01)` — $g=\nabla L$, $s=\mathrm{CG}(F,g)$, $\beta=\sqrt{2\delta/s^\top Fs}$; çizgi araması $\beta\cdot0.5^k$: KL ≤ 1.5δ ve L iyileşmişse kabul, yoksa eski parametreler. Kabul edildiyse True.
- `train_trpo(iters=60, max_kl=0.01, seed=0)` — politika TRPO ile, değer Adam (1e-3, 4 epoch).

In [ ]:
%%writefile stage_d.py
import math
import torch
import torch.nn as nn

G_, MC, MP, L, FM, TAU = 9.8, 1.0, 0.1, 0.5, 10.0, 0.02


def cartpole_step(x, a):
    pos, vel, th, om = x.unbind(1)
    f = torch.where(a == 1, FM, -FM).to(x.dtype)
    ct, st = torch.cos(th), torch.sin(th)
    tm, pml = MC + MP, MP * L
    temp = (f + pml * om ** 2 * st) / tm
    thacc = (G_ * st - ct * temp) / (L * (4.0 / 3.0 - MP * ct ** 2 / tm))
    xacc = temp - pml * thacc * ct / tm
    x2 = torch.stack([pos + TAU * vel, vel + TAU * xacc, th + TAU * om, om + TAU * thacc], 1)
    done = (x2[:, 0].abs() > 2.4) | (x2[:, 2].abs() > 12 * 2 * math.pi / 360)
    return x2, done


def evaluate(net, n=20, max_len=500, seed=123):
    g = torch.Generator().manual_seed(seed)
    x = (torch.rand(n, 4, generator=g) - 0.5) * 0.1
    alive = torch.ones(n, dtype=torch.bool)
    lens = torch.zeros(n)
    for _ in range(max_len):
        with torch.no_grad():
            x, d = cartpole_step(x, net(x)[0].argmax(1))
        lens += alive.float()
        alive &= ~d
    return lens


class PolicyValue(nn.Module):
    def __init__(self):
        super().__init__()
        self.pi = nn.Sequential(nn.Linear(4, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 2))
        self.v = nn.Sequential(nn.Linear(4, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 1))

    def forward(self, x):
        return self.pi(x), self.v(x).squeeze(-1)


def rollout(net, state, g, n_envs=16, T=128):
    """state = [x, t_ep, lens] (yerinde güncellenir). Döner: X, A, LP, V, R, D, v_last."""
    x, t_ep, lens = state
    X, A, LP, V, R, D = [], [], [], [], [], []
    with torch.no_grad():
        for _ in range(T):
            lo, v = net(x)
            dist = torch.distributions.Categorical(logits=lo)
            a = dist.sample()
            x2, d = cartpole_step(x, a)
            t_ep += 1
            X.append(x); A.append(a); LP.append(dist.log_prob(a)); V.append(v); R.append(torch.ones(n_envs)); D.append(d.float())
            reset = d | (t_ep >= 500)
            if reset.any():
                lens += t_ep[reset].tolist()
                t_ep[reset] = 0
                x2 = torch.where(reset[:, None], (torch.rand(n_envs, 4, generator=g) - 0.5) * 0.1, x2)
            x = x2
        v_last = net(x)[1]
    state[0] = x
    return torch.stack(X), torch.stack(A), torch.stack(LP), torch.stack(V), torch.stack(R), torch.stack(D), v_last


def gae(r, v, d, v_last, gamma, lam):
    ...


def ppo_loss(logp_new, logp_old, adv, eps):
    ...


def train_ppo(iters=60, eps=0.2, epochs=4, mb=256, lr=3e-4, seed=0):
    ...


def fvp(pi, X, v, damping=0.1):
    ...


def cg(Avp, b, iters=10):
    ...


def trpo_step(pi, X, A, adv, old_logp, max_kl=0.01):
    ...


def train_trpo(iters=60, max_kl=0.01, seed=0):
    ...


net, lens = train_ppo(5)
print(evaluate(net).mean())

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

**GAE: döngü mü matris mi?** PyTorch, `r, v, d (T, N)`, `v_last (N,)`.

- `gae_scan(r, v, d, v_last, gamma, lam)` — zamanda geriye tarama, ortamlar paralel (T adımlık döngü).
- `gae_matrix(r, v, d, v_last, gamma, lam)` — döngüsüz: $A=M\delta$, $M_{t,k}=(\gamma\lambda)^{k-t}$ ($k\ge t$ ve $[t,k)$ aralığında bitiş yoksa), ortam başına `(N, T, T)` ve `einsum`. Bitişler: `d`'nin dışlayıcı kümülatif toplamı aynı olmalı.
- `walk_batch(N, T, g)` — 19 durumlu rastgele yürüyüşten N bölüm (merkezden, ±1, uçlarda ±1 ödül), bittikten sonra dolgu: `(S (T+1, N), R, D)`.

In [ ]:
%%writefile stage_e.py
import torch


def gae_scan(r, v, d, v_last, gamma, lam):
    ...


def gae_matrix(r, v, d, v_last, gamma, lam):
    ...


def walk_batch(N, T, g, n=19):
    ...


g = torch.Generator().manual_seed(0)
S, R, D = walk_batch(4, 50, g)
print(S.shape, R.sum(0))

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Hessian tanımsızken: Newton eyer noktasına, güven bölgesi minimuma

$f(x,y)=x^2-y^2+\tfrac14y^4$: (0,0) eyer noktası, minimumlar $(0,\pm\sqrt2)$, $f=-1$. Başlangıç (1, 0.01).

- `newton(p0, n)` — saf Newton $p\leftarrow p-H^{-1}g$.
- `trust_region_saddle(p0, n, delta0=0.5)` — b'deki dogleg/Cauchy ve ρ kuralları.

In [ ]:
%%writefile extra_x1.py
import math


def f(p):
    x, y = p
    return x * x - y * y + 0.25 * y ** 4


def grad(p):
    ...


def hess(p):
    ...


def newton(p0, n):
    ...


def trust_region_saddle(p0, n, delta0=0.5):
    ...


print(newton([1.0, 0.01], 20)[-1], trust_region_saddle([1.0, 0.01], 50)[-1])

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · PPO'nun kırpılmış vekili: kötümser alt sınır

Tek durum, 3 eylem, eski politika $\pi_0$, avantajlar A. Yön $d$ boyunca $\theta=\theta_0+t\,d$:

- `surrogates(t_grid, theta0, d, A, eps=0.2, beta=1.0)` → `(L_CPI, L_CLIP, L_KLPEN, KL)` her biri `t_grid` uzunluğunda: $L^{CPI}=\sum_a\pi_0(a)r_a A_a$, $L^{CLIP}=\sum_a\pi_0(a)\min(r_aA_a,\mathrm{clip}(r_a)A_a)$, $L^{KLPEN}=L^{CPI}-\beta\,\mathrm{KL}(\pi_0\|\pi)$.

In [ ]:
%%writefile extra_x2.py
import numpy as np


def surrogates(t_grid, theta0, d, A, eps=0.2, beta=1.0):
    ...


t = np.linspace(-3, 3, 7)
print(surrogates(t, np.zeros(3), np.array([1.0, -1.0, 0.0]), np.array([1.0, -0.5, 0.0])))

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · TRPO'nun monoton iyileşme sınırı

Küçük bir MDP'de (S = 6, A = 3, γ = 0.9) $\pi$ ve ondan türetilen $\tilde\pi=(1-\alpha)\pi+\alpha\pi'$ karışımları:

- `eta(P, R, d0, pi, gamma)` — gerçek indirimli getiri.
- `surrogate_L(P, R, d0, pi, pi_new, gamma)` — $\eta(\pi)+\sum_s\rho_\pi(s)\sum_a\tilde\pi(a|s)A_\pi(s,a)$ ($\rho_\pi$ indirimli ziyaret).
- `tv_max(pi, pi_new)`, `kl_max(pi, pi_new)`.
- `bound(P, R, d0, pi, pi_new, gamma)` — $L-\dfrac{4\epsilon\gamma}{(1-\gamma)^2}\mathrm{KL}^{\max}$, $\epsilon=\max|A_\pi|$.

In [ ]:
%%writefile extra_x3.py
import numpy as np


def eta(P, R, d0, pi, gamma):
    ...


def surrogate_L(P, R, d0, pi, pi_new, gamma):
    ...


def kl_max(pi, pi_new):
    ...


def bound(P, R, d0, pi, pi_new, gamma):
    ...


rng = np.random.default_rng(0)
P = rng.random((3, 6, 6)); P /= P.sum(-1, keepdims=True); R = rng.random((6, 3)); d0 = np.full(6, 1 / 6)
pi = np.full((6, 3), 1 / 3)
print(eta(P, R, d0, pi, 0.9))

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Schulman ve ark. (2015): TRPO

[Schulman ve ark. — Trust Region Policy Optimization (ICML 2015)](https://arxiv.org/abs/1502.05477)

d aşamasındaki TRPO'yu sürekli eylemler için Gauss politikasına genişlet (çapraz kovaryanslı; KL kapalı form) ve MuJoCo'da (Hopper, Walker2d — Gymnasium) "single path" ve "vine" örneklemesi olmadan tek yol sürümüyle çalıştır. Makalenin Ek C'sindeki Fisher-vektör çarpımı hilesini (analitik KL Hessian'ı) uygula ve otomatik türevli sürümle hız/sonuç karşılaştırması yap; δ ∈ {0.001, 0.01, 0.05} taraması.


**Adımlar**

1. CG 10 yineleme, sönümleme 0.1; çizgi aramasında KL ve vekil iyileşme koşulu.
2. Doğal gradyan (çizgi araması olmadan sabit adım) ile karşılaştır.

**Kabul ölçütleri**

- TRPO sabit adımlı doğal gradyandan daha kararlı (makale Şekil 4 nitel).
- Ortalama KL her güncellemede δ'nın ~1.5 katının altında.

## Makale ödevi: Schulman ve ark. (2017): PPO

[Schulman ve ark. — Proximal Policy Optimization Algorithms (2017)](https://arxiv.org/abs/1707.06347)

Makalenin Tablo 1 karşılaştırmasını küçük ölçekte yeniden üret: kırpmasız, kırpılmış (ε = 0.1, 0.2, 0.3), sabit KL cezası (β = 0.3, 1, 3), uyarlamalı KL cezası (d_targ = 0.003, 0.01, 0.03) — CartPole, Acrobot ve bir MuJoCo ortamında 3 tohum. Ayrıca "37 uygulama ayrıntısı"ndan (Huang ve ark. 2022) avantaj normalizasyonu, değer kırpma ve öğrenme oranı azaltmanın ablation'ını yap.


**Adımlar**

1. d aşamasındaki ppo_loss'u ceza seçenekleriyle genişlet.
2. e aşamasındaki GAE'yi λ ∈ {0.9, 0.95, 0.99} ile tara.

**Kabul ölçütleri**

- Kırpılmış ε = 0.2 en iyi ya da en iyiye yakın (makale Tablo 1 nitel).
- Kırpmasız sürüm bazı tohumlarda çöker.

## Kaynaklar

- [Schulman ve ark. — Trust Region Policy Optimization (ICML 2015)](https://arxiv.org/abs/1502.05477) — arXiv
- [Schulman ve ark. — Proximal Policy Optimization Algorithms (2017)](https://arxiv.org/abs/1707.06347) — arXiv
- [Schulman ve ark. — High-Dimensional Continuous Control Using Generalized Advantage Estimation (ICLR 2016)](https://arxiv.org/abs/1506.02438) — arXiv
- [Kakade — A Natural Policy Gradient (NeurIPS 2001)](https://papers.nips.cc/paper/2001/hash/4b86abe48d358ecf194c56c69108433e-Abstract.html) — ©
- [Kakade & Langford — Approximately Optimal Approximate Reinforcement Learning (ICML 2002)](https://dl.acm.org/doi/10.5555/645531.656005) — ©
- [Nocedal & Wright — Numerical Optimization (2. baskı), Bölüm 4](https://doi.org/10.1007/978-0-387-40065-5) — ©